# Phase 0 Colab runner

Thin launcher: all logic lives in `src/`. Runtime: GPU (T4 is fine for ViT-B/16).

One-time setup: create a fine-grained GitHub token for this repo only (Contents: read and write) and add it in Colab Secrets (key icon) as `GITHUB_TOKEN`.

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')

REPO = 'sohilkhan-2611/-clip-zero-shot-forgetting'
BRANCH = 'phase-0'
CKPT_DIR = '/content/drive/MyDrive/clip-forgetting/checkpoints'
DATA_DIR = '/content/data'   # local disk is much faster than Drive
token = userdata.get('GITHUB_TOKEN')

In [ ]:
import os
if not os.path.exists('/content/repo'):
    !git clone -q -b {BRANCH} https://{token}@github.com/{REPO}.git /content/repo
%cd /content/repo
!git pull -q
!pip install -q -e .[dev]
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Sanity tests (CPU, random weights, about 1 minute)

In [ ]:
!python -m pytest -q

## 2. Smoke run: zero-shot on 256 images per set
Catches download or path problems before the long run. These numbers are noisy, do not record them.

In [ ]:
!python -m src.evaluate --config configs/eval_zeroshot_vitb16.yaml --set data.root={DATA_DIR} data.max_samples=256 data.eval_sets=[flowers,cifar10,dtd]

## 3. Full zero-shot eval of the untouched model (done-when check 1)
Prints ours vs published per dataset. Food101 is a ~5 GB download; drop it from `eval_sets` for a first pass if needed.

In [ ]:
!python -m src.evaluate --config configs/eval_zeroshot_vitb16.yaml --set data.root={DATA_DIR}

## 4. One fine-tune on Flowers102 (done-when check 2)
Hyperparameters in the config are placeholders until the Liao et al. recipe is filled in.

In [ ]:
!python -m src.finetune --config configs/finetune_flowers_vitb16.yaml --set data.root={DATA_DIR} output.checkpoint_dir={CKPT_DIR}

## 5. Push results JSON back to the repo

In [ ]:
!git config user.name 'colab-runner'
!git config user.email 'colab-runner@users.noreply.github.com'
!git add results/
!git commit -m 'Phase 0 results from Colab' || echo 'nothing to commit'
!git push -q origin {BRANCH}